In [16]:
import json
from tqdm import tqdm
import spacy

In [17]:
nlp = spacy.blank("en")

# BIO-TAGGING:
def get_bio_tags(review_text, annotations):
    doc = nlp(review_text)
    tokens = [t.text for t in doc]
    tags = ["O"] * len(tokens)

    for ann in annotations:
        aspect = ann['aspect'].lower()
        sentiment = ann['sentiment']

        label = "POS" if sentiment == 1 else "NEG" if sentiment == -1 else "NEU"

        aspect_tokens = [t.text for t in nlp(aspect.lower())]
        n_aspect = len(aspect_tokens)

        for i in range(len(tokens) - n_aspect + 1):
            if [t.lower() for t in tokens[i:i+n_aspect]] == aspect_tokens:
                tags[i] = f"B-{label}"
                for j in range(1, n_aspect):
                    tags[i+j] = f"I-{label}"
                break
    return tokens, tags

In [18]:
review_samples = []
with open('absa_checkpoints.jsonl', 'r') as f:
    for line in tqdm(f, desc="Converting to BIO"):
        item = json.loads(line)
        tokens, tags = get_bio_tags(item['review'], item['annotations'])
        review_samples.append((tokens, tags))
review_samples[0]

Converting to BIO: 5248it [00:05, 988.21it/s] 


(['put',
  'this',
  'movie',
  'out',
  'of',
  'its',
  'misery',
  'and',
  'burn',
  'the',
  'negatives',
  'what',
  'am',
  'i',
  'saying',
  'the',
  'whole',
  'movie',
  'was',
  'negative',
  'fortunately',
  'only',
  'a',
  'very',
  'few',
  'would',
  'find',
  'this',
  'movie',
  'the',
  'least',
  'bit',
  'appealing',
  'this',
  'is',
  'what',
  'the',
  'vast',
  'american',
  'majority',
  'would',
  'call',
  'too',
  'much',
  'sex',
  'and',
  'violence',
  'it',
  'will',
  'probably',
  'show',
  'up',
  'on',
  'some',
  'nonpremium',
  'cable',
  'channel',
  'someday',
  'just',
  'for',
  'the',
  'shock',
  'value',
  'but',
  'after',
  'editing',
  'out',
  'the',
  'nudity',
  'most',
  'of',
  'the',
  'violence',
  'will',
  'stay',
  'all',
  'that',
  'will',
  'be',
  'left',
  'is',
  'minutes',
  'of',
  'really',
  'bad',
  'acting',
  'interspersed',
  'with',
  'minutes',
  'of',
  'commercials',
  'there',
  'are',
  'just',
  'too',
  '

In [19]:
tag2idx = {
    "O": 0,
    "B-POS": 1, "I-POS": 2,
    "B-NEG": 3, "I-NEG": 4,
    "B-NEU": 5, "I-NEU": 6
}
num_tags = len(tag2idx)

In [2]:
!pip install pytorch-crf

In [21]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchcrf import CRF
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from transformers import XLMRobertaTokenizer, XLMRobertaModel

In [22]:
from sklearn.model_selection import train_test_split
train_data,test_data = train_test_split(review_samples,test_size=0.2, random_state=42)
print(f"Training on {len(train_data)} reviews")
print(f"Testing on {len(test_data)} reviews")

Training on 4198 reviews
Testing on 1050 reviews


In [23]:
class XLMR_ABSADataset(Dataset):
    def __init__(self, data_samples, tokenizer, tag2idx, max_len=128):
        self.data = data_samples
        self.tokenizer = tokenizer
        self.tag2idx = tag2idx
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        tokens, tags = self.data[idx]

        encoding = self.tokenizer(
            tokens,
            is_split_into_words=True,
            return_offsets_mapping=True,
            padding='max_length',
            truncation=True,
            max_length=self.max_len
        )

        labels = []
        word_ids = encoding.word_ids()

        previous_word_idx = None
        for word_idx in word_ids:
            if word_idx is None:
                labels.append(-100)
            elif word_idx != previous_word_idx:
                labels.append(self.tag2idx[tags[word_idx]])
            else:
                labels.append(-100)
            previous_word_idx = word_idx

        return {
            'input_ids': torch.tensor(encoding['input_ids'], dtype=torch.long),
            'attention_mask': torch.tensor(encoding['attention_mask'], dtype=torch.long),
            'labels': torch.tensor(labels, dtype=torch.long)
        }

In [25]:
tokenizer = XLMRobertaTokenizer.from_pretrained('xlm-roberta-base')

train_ds = XLMR_ABSADataset(train_data, tokenizer, tag2idx)
test_ds = XLMR_ABSADataset(test_data, tokenizer, tag2idx)

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=16, shuffle=False)

In [26]:
class XLMR_BiLSTM_CRF(nn.Module):
    def __init__(self, num_tags, hidden_dim, freeze_xlmr=True):
        super(XLMR_BiLSTM_CRF, self).__init__()
        self.xlmr = XLMRobertaModel.from_pretrained('xlm-roberta-base')

        if freeze_xlmr:
            for param in self.xlmr.parameters():
                param.requires_grad = False
        # Input is 768 (XLM-R hidden size)
        self.lstm = nn.LSTM(768, hidden_dim // 2,
                            num_layers=1, bidirectional=True, batch_first=True)

        self.dropout = nn.Dropout(0.5)
        self.hidden2tag = nn.Linear(hidden_dim, num_tags)
        self.crf = CRF(num_tags, batch_first=True)

    def forward(self, input_ids, attention_mask, labels=None):
            outputs = self.xlmr(input_ids=input_ids, attention_mask=attention_mask)
            sequence_output = outputs.last_hidden_state

            lstm_out, _ = self.lstm(sequence_output)
            lstm_out = self.dropout(lstm_out)
            emissions = self.hidden2tag(lstm_out)

            if labels is not None:
                active_mask = (attention_mask.bool()) & (labels != -100)
                active_mask[:, 0] = True
                clean_labels = labels.clone()
                clean_labels[labels == -100] = 0

                return -self.crf(emissions, clean_labels, mask=active_mask)
            else:
                mask = attention_mask.bool()
                return self.crf.decode(emissions, mask=mask)

In [27]:
hidden_dim = 128
model = XLMR_BiLSTM_CRF(num_tags,hidden_dim)
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                       lr=1e-3, weight_decay=1e-4)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [12]:
def train_validate(model,train_loader,test_loader,epochs=5):
  best_val_loss = float('inf')
  for epoch in range(epochs):
      model.train()
      total_train_loss = 0

      for batch in train_loader:
          ids = batch['input_ids'].to(device)
          mask = batch['attention_mask'].to(device)
          targets = batch['labels'].to(device)

          optimizer.zero_grad()
          loss = model(ids, mask, labels=targets)
          loss.backward()

          torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
          optimizer.step()
          total_train_loss += loss.item()

      #VALIDATION PHASE
      model.eval()
      total_val_loss = 0
      with torch.no_grad():
          for batch in test_loader:
              ids = batch['input_ids'].to(device)
              mask = batch['attention_mask'].to(device)
              targets = batch['labels'].to(device)

              loss = model(ids, mask, labels=targets)
              total_val_loss += loss.item()

      avg_train_loss = total_train_loss / len(train_loader)
      avg_val_loss = total_val_loss / len(test_loader)

      print(f"Epoch {epoch+1} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")


      if avg_val_loss < best_val_loss:
          best_val_loss = avg_val_loss
          torch.save(model.state_dict(), "best_xlmr_absa_model2.bin")
          print("--> Best Model Saved!")

In [48]:
train_validate(model,train_loader,test_loader,epochs=5)

Epoch 1 | Train Loss: 225.3883 | Val Loss: 133.6770
--> Best Model Saved!
Epoch 2 | Train Loss: 135.0781 | Val Loss: 112.8992
--> Best Model Saved!
Epoch 3 | Train Loss: 121.5508 | Val Loss: 107.1702
--> Best Model Saved!
Epoch 4 | Train Loss: 112.4677 | Val Loss: 103.4235
--> Best Model Saved!
Epoch 5 | Train Loss: 105.3688 | Val Loss: 92.8082
--> Best Model Saved!


In [30]:
from seqeval.metrics import classification_report,f1_score

In [31]:
def evaluate_model(model, data_loader, idx2tag):
    model.eval()
    y_true = []
    y_pred = []

    with torch.no_grad():
        for batch in data_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)


            paths = model(input_ids, attention_mask)

            for i in range(len(paths)):
                label_list = labels[i].cpu().numpy()
                pred_list = paths[i]

                valid_indices = [j for j, l in enumerate(label_list) if l != -100]

                true_tags = [idx2tag[label_list[j]] for j in valid_indices]
                pred_tags = [idx2tag[pred_list[j]] for j in valid_indices]

                y_true.append(true_tags)
                y_pred.append(pred_tags)

    print(classification_report(y_true, y_pred, zero_division=0))
    return f1_score(y_true, y_pred)

In [50]:
model.load_state_dict(torch.load("best_xlmr_absa_model.bin"))
idx2tag = {v: k for k, v in tag2idx.items()}
f1 = evaluate_model(model, test_loader, idx2tag)

              precision    recall  f1-score   support

         NEG       0.52      0.34      0.41       792
         NEU       0.00      0.00      0.00        19
         POS       0.63      0.25      0.36      1166

   micro avg       0.57      0.28      0.38      1977
   macro avg       0.38      0.20      0.26      1977
weighted avg       0.58      0.28      0.37      1977



In [54]:
model.eval()
batch = next(iter(test_loader))
ids = batch['input_ids'][:1].to(device)
mask = batch['attention_mask'][:1].to(device)
labels = batch['labels'][:1]

with torch.no_grad():
    preds = model(ids, mask)[0]

    # Filter out -100 to see actual word-level alignment
    valid_preds = [idx2tag[p] for p, l in zip(preds, labels[0]) if l != -100]
    valid_labels = [idx2tag[l.item()] for l in labels[0] if l != -100]

    print(f"Actual Words: {valid_labels}")
    print(f"Model Predicted: {valid_preds}")

Actual Words: ['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-POS', 'O', 'O', 'O', 'B-POS', 'I-POS', 'O', 'B-POS', 'O', 'O', 'O', 'O']
Model Predicted: ['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-POS', 'O', 'O', 'O', 'B-POS', 'I-POS', 'O', 'B-POS', 'O', 'O', 'O', 'O']


In [6]:
def predict_xlmr_absa(text, model, tokenizer, idx2tag, device):
    model.eval()

    encoding = tokenizer(
        text,
        return_offsets_mapping=True,
        padding='max_length',
        truncation=True,
        max_length=128,
        return_tensors="pt"
    )

    ids = encoding['input_ids'].to(device)
    mask = encoding['attention_mask'].to(device)

    word_ids = encoding.word_ids(batch_index=0)

    with torch.no_grad():

        paths = model(ids, mask)
        path = paths[0]


    results = []
    tokens = tokenizer.convert_ids_to_tokens(ids[0])

    previous_word_idx = None
    for i, word_idx in enumerate(word_ids):

        if word_idx is not None and word_idx != previous_word_idx:
            tag = idx2tag[path[i]]

            word = tokens[i].replace(' ', '')
            results.append((word, tag))
        previous_word_idx = word_idx

    return results

raw_review = "the camera of this phone was too good"
predictions = predict_xlmr_absa(raw_review, model, tokenizer, idx2tag, device)

for word, tag in predictions:
    if tag != "O":
        print(f"{word:15} -> {tag}")
    else:
        print(f"{word:15} -> {tag}")

NameError: name 'model' is not defined

In [82]:
# phase2 training
for param in model.xlmr.parameters():
    param.requires_grad = True

optimizer = optim.AdamW([
    {'params': model.xlmr.parameters(), 'lr': 2e-5},
    {'params': model.lstm.parameters(), 'lr': 1e-3},
    {'params': model.hidden2tag.parameters(), 'lr': 1e-3},
    {'params': model.crf.parameters(), 'lr': 1e-3}
], weight_decay=1e-4)

train_validate(model,train_loader,test_loader,epochs=5)

Epoch 1 | Train Loss: 95.6892 | Val Loss: 78.5545
--> Best Model Saved!
Epoch 2 | Train Loss: 74.1430 | Val Loss: 72.2769
--> Best Model Saved!
Epoch 3 | Train Loss: 58.0651 | Val Loss: 73.0277
Epoch 4 | Train Loss: 43.5552 | Val Loss: 79.5022
Epoch 5 | Train Loss: 29.9482 | Val Loss: 81.2024


In [32]:
model.load_state_dict(torch.load("best_xlmr_absa_model2.bin"))
# f1_2 = evaluate_model(model, test_loader, idx2tag)

<All keys matched successfully>

In [42]:
raw_review = "mne ye product kal order liya tha and today it has arrived do heads off to the delivery so experience was good"
predictions = predict_xlmr_absa(raw_review, model, tokenizer, idx2tag, device)

for word, tag in predictions:
    if tag != "O":
        print(f"{word:15} -> {tag}")
    else:
        print(f"{word:15} -> {tag}")

▁mne            -> O
▁ye             -> O
▁product        -> O
▁kal            -> O
▁order          -> O
▁liya           -> O
▁tha            -> O
▁and            -> O
▁today          -> O
▁it             -> O
▁has            -> O
▁arrived        -> O
▁do             -> O
▁head           -> O
▁off            -> O
▁to             -> O
▁the            -> O
▁delivery       -> B-POS
▁so             -> O
▁experience     -> O
▁was            -> O
▁good           -> O
